# V1-18: 기업 식별 결합 정책 로컬 Validation

이 노트북은 개별 검색어 확장이 아니라, `exact + fuzzy + LLM 검색어 가설 + 재질문 + NOT_FOUND`를 합친 기업 식별 정책을 비교한다.

- **A. 결정론적 정책**: raw exact, fuzzy, cutoff, 재질문만 사용
- **B. 로컬 LLM 확장 정책**: A에 미해결 사례의 Qwen 검색어 가설을 추가

> 이 평가는 반복 개선용 **local validation**이다. 운영 후보가 OpenAI API이므로, 여기서 선택한 정책은 변경 없이 별도의 API holdout에서 한 번 재검증해야 한다. LLM은 검색어만 제안하며 `corp_code`를 선택하거나 새 후보를 자동 확정하지 않는다.

## 테스트 목적과 해석

이 테스트로 확인하려는 것은 ‘Qwen이 기업명을 얼마나 잘 아는가’ 하나가 아니다. 후보를 넓히는 LLM 경로를 붙였을 때 다음 안전 정책이 유지되는지 확인한다.

1. 고유 raw exact만 자동 확정한다.
2. LLM이 새로 찾은 실제 기업은 항상 `NEEDS_CONFIRMATION`으로 보낸다.
3. 포괄 그룹명은 LLM을 호출하지 않고 재질문한다.
4. 후보가 없고 최고 fuzzy 점수가 임시 cutoff 75보다 낮으면 `NOT_FOUND`로 보낸다.

핵심 지표는 Candidate Recall, Unsafe Selection Rate, State Accuracy, Not-found Recall, False Not-found Rate다. B가 후보 회수를 올리더라도 자동 오선택이나 유효 기업의 `NOT_FOUND`를 늘리면 채택하지 않는다.

In [ ]:
import io
from datetime import datetime, timezone
import json
from math import ceil
import os
from pathlib import Path
import re
from statistics import median
import sys
import time
from typing import Literal
import xml.etree.ElementTree as ET
import zipfile

import pandas as pd
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')
for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

if not os.environ.get('OPENDART_API_KEY'):
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

api_key = os.environ['OPENDART_API_KEY']
LOCAL_MODEL = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b')
OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
TEMPERATURE, REQUEST_TIMEOUT_S = 0, 300
FUZZY_KEY_LIMIT, FUZZY_RESULT_LIMIT = 50, 5
PROVISIONAL_CUTOFF = 75

print(f'로컬 모델: {LOCAL_MODEL} / temperature={TEMPERATURE} / reasoning=False')
print(f'임시 fuzzy cutoff: {PROVISIONAL_CUTOFF} (이번 모델 비교에서는 고정)')
print('GPU·num_ctx·keep_alive 등 Ollama runner 옵션은 지정하지 않습니다.')

In [ ]:
sys.path.insert(0, str(project_root / 'src'))
from dart.client import CORP_CODE_URL, _get_bytes


def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(stream):
        raise RuntimeError(zip_bytes.decode('utf-8', errors='replace'))
    with zipfile.ZipFile(stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])
    root = ET.fromstring(xml_bytes)
    return [{child.tag: child.text or '' for child in item} for item in root.findall('list')]


def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())


def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(values.values()) for key, values in index.items()}


corp_records = load_corp_records(api_key)
name_index = build_name_index(corp_records)
name_keys = list(name_index)
retrieved_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(f'OpenDART 목록 조회 시각: {retrieved_at}')
print(f'법인 레코드: {len(corp_records):,} / 정규화 이름 키: {len(name_keys):,}')

In [ ]:
# 이전 05~17번의 입력 문자열과 구분한 개발용 policy validation 사례다.
# label은 평가용이며 LLM prompt에는 전달하지 않는다.
CASES = [
    ('롯데케미칼 2024년 매출을 알려줘.', '롯데케미칼', ['롯데케미칼'], 'RESOLVED', '정식명'),
    ('롯데케미컬 최근 실적을 알려줘.', '롯데케미컬', ['롯데케미칼'], 'NEEDS_CONFIRMATION', '한글/영문 표기'),
    ('삼성 SDI 영업이익은?', '삼성 SDI', ['삼성SDI'], 'RESOLVED', '공백 표기'),
    ('삼성에스디아이 매출 추세를 보여줘.', '삼성에스디아이', ['삼성SDI'], 'NEEDS_CONFIRMATION', '한글 음역'),
    ('SK 이노베이션의 재무를 조회해줘.', 'SK 이노베이션', ['SK이노베이션'], 'RESOLVED', '공백 표기'),
    ('에스케이이노베이션 실적을 요약해줘.', '에스케이이노베이션', ['SK이노베이션'], 'NEEDS_CONFIRMATION', '한글 음역'),
    ('두산 에너빌리티 매출은?', '두산 에너빌리티', ['두산에너빌리티'], 'RESOLVED', '공백 표기'),
    ('한화 오션 최근 실적은?', '한화 오션', ['한화오션'], 'RESOLVED', '공백 표기'),
    ('현대 모비스 영업이익을 알려줘.', '현대 모비스', ['현대모비스'], 'RESOLVED', '공백 표기'),
    ('현대모비수 매출을 보여줘.', '현대모비수', ['현대모비스'], 'NEEDS_CONFIRMATION', '오타'),
    ('아모레퍼시퍽 실적을 알려줘.', '아모레퍼시퍽', ['아모레퍼시픽'], 'NEEDS_CONFIRMATION', '오타'),
    ('대한항콩 영업이익은?', '대한항콩', ['대한항공'], 'NEEDS_CONFIRMATION', '오타'),
    ('고려아연 재무정보를 알려줘.', '고려아연', ['고려아연'], 'RESOLVED', '정식명'),
    ('롯데 계열사 실적을 알려줘.', '롯데', [], 'NEEDS_CONFIRMATION', '포괄 그룹명'),
    ('GS 최근 실적을 알려줘.', 'GS', [], 'NEEDS_CONFIRMATION', '포괄 그룹명'),
    ('신세계의 매출을 알려줘.', '신세계', [], 'NEEDS_CONFIRMATION', '포괄 그룹명'),
    ('CJ 실적을 분석해줘.', 'CJ', [], 'NEEDS_CONFIRMATION', '포괄 그룹명'),
    ('새봄테크2027의 매출을 알려줘.', '새봄테크2027', [], 'NOT_FOUND', '미등록 가상명'),
    ('오로라스페이스2028 실적을 알려줘.', '오로라스페이스2028', [], 'NOT_FOUND', '미등록 가상명'),
    ('동해바이오솔루션2040 매출을 알려줘.', '동해바이오솔루션2040', [], 'NOT_FOUND', '미등록 가상명'),
    ('한화오션솔루션즈 실적을 알려줘.', '한화오션솔루션즈', [], 'NOT_FOUND', '실제 기업 유사 미등록명'),
]
TEST_CASES = [{'질문': question, '원문 기업 표현': raw, '기대 대상': targets, '기대 상태': state, '유형': category} for question, raw, targets, state, category in CASES]

TARGETS = sorted({target for case in TEST_CASES for target in case['기대 대상']})
target_records = {}
for target in TARGETS:
    matches = [record for record in corp_records if normalize_name(record['corp_name']) == normalize_name(target) and record['stock_code'].strip()]
    if len(matches) != 1:
        raise AssertionError(f'{target}: 상장 OpenDART 법인 검증 실패 ({len(matches)}건)')
    target_records[target] = matches[0]

for case in TEST_CASES:
    if case['기대 상태'] == 'NOT_FOUND' and name_index.get(normalize_name(case['원문 기업 표현'])):
        raise AssertionError(f"{case['원문 기업 표현']}: 미등록 가상명 exact 검증 실패")

pd.DataFrame(TEST_CASES)[['원문 기업 표현', '유형', '기대 상태', '기대 대상']]

In [ ]:
def exact_candidates(query, source='raw_exact'):
    return {record['corp_code']: {'record': record, 'sources': {source}, 'score': None} for record in name_index.get(normalize_name(query), [])}


def fuzzy_candidates(query, source='raw_fuzzy'):
    scored = process.extract(normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT)
    candidates = {}
    for key, score, _ in scored:
        for record in name_index[key]:
            current = candidates.get(record['corp_code'])
            if current is None or score > current['score']:
                candidates[record['corp_code']] = {'record': record, 'sources': {source}, 'score': score}
    return dict(sorted(candidates.items(), key=lambda item: (item[1]['score'], bool(item[1]['record']['stock_code'].strip()), item[1]['record']['modify_date']), reverse=True)[:FUZZY_RESULT_LIMIT])


def merge_candidates(*candidate_sets):
    merged = {}
    for candidate_set in candidate_sets:
        for corp_code, candidate in candidate_set.items():
            existing = merged.setdefault(corp_code, {'record': candidate['record'], 'sources': set(), 'score': candidate['score']})
            existing['sources'].update(candidate['sources'])
            if candidate['score'] is not None:
                existing['score'] = max(existing['score'] or 0, candidate['score'])
    return merged


# 대규모 별칭 사전이 아니라, 단일 법인으로 자동 확정하면 안 되는 포괄 그룹 표현만 보호한다.
GENERIC_GROUP_TERMS = {normalize_name(name) for name in ('롯데', 'GS', '신세계', 'CJ')}


def candidate_display(candidates):
    return [f"{item['record']['corp_name']} ({item['record']['stock_code'].strip() or '비상장'})" for item in candidates.values()]

In [ ]:
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태')
    search_terms: list[str] = Field(default_factory=list, max_length=2, description='OpenDART에서 검증할 정식 기업명 검색 가설. corp_code·종목코드·설명 문장은 금지.')


DETAILED_PROMPT = '''
역할: 당신은 한국 기업 리서치 시스템의 'OpenDART 기업명 검색어 계획기'다. 기업을 확정하거나 corp_code를 고르는 역할이 아니다. 프로그램은 원문 기업 표현의 OpenDART exact 결과가 고유하지 않을 때만 호출한다.

목표: 사용자 질문의 원문 기업 표현을 OpenDART 기업 목록에서 다시 찾기 위한 검색 가설을 최대 2개 만든다. 제안은 후보 탐색 입력일 뿐이며, 뒤의 프로그램이 OpenDART 목록으로 존재 여부를 검증하고 필요하면 사용자에게 재질문한다.

candidate를 선택할 조건:
1. 원문이 특정 기업을 가리킨다고 볼 만한 널리 쓰이는 약칭·축약·브랜드식 표기인 경우, 그 기업의 정식 법인명 또는 공식 영문 표기를 검색 가설로 제안한다.
2. 띄어쓰기, 대소문자, 한글 음역/영문 공식 표기 차이, 한두 글자의 명백한 오타처럼 표면 형태만 달라도 특정 기업을 가리킬 근거가 충분한 경우, OpenDART에 있을 법한 표준 기업명으로 바꿔 제안한다.
3. 원문을 그대로 반복하는 것은 그 표현 자체가 정식 기업명 검색어로 유효할 가능성이 있을 때만 허용한다. 별칭·오타·음역 표기라면 같은 문자열을 반복하지 말고 표준 검색 가설을 제안한다.

ambiguous 또는 unknown을 선택할 조건:
1. 그룹명, 업종명, 보통명사, 한 단어가 여러 독립 기업을 가리킬 수 있는 표현처럼 특정 법인을 지목할 근거가 부족하면 ambiguous와 빈 목록을 반환한다.
2. 존재하지 않는 것으로 보이거나 임의 문자열이며 특정 기업으로 연결할 근거가 없으면 unknown과 빈 목록을 반환한다.

금지 규칙:
- 사용자가 언급하지 않은 계열사, 경쟁사, 유사 업종 회사를 연상으로 추가하지 않는다.
- 확신이 없다는 이유로 그럴듯한 실제 기업명을 임의로 만들거나, raw 표현과 무관한 회사를 제안하지 않는다.
- corp_code, 종목코드, 거래소 코드, 투자 의견, 근거 설명, 문장을 search_terms에 넣지 않는다.
- 최종 선택·순위·확정이라는 표현을 하지 않는다.

출력 계약: decision은 candidate, ambiguous, unknown 중 하나다. candidate일 때만 공백을 제외한 기업명 검색어를 1~2개 반환한다. ambiguous와 unknown은 반드시 search_terms=[]를 반환한다. 설명 없이 JSON 객체만 반환한다.
'''
SCHEMA_JSON = json.dumps(EntitySearchPlan.model_json_schema(), ensure_ascii=False)
qwen_llm = ChatOllama(model=LOCAL_MODEL, base_url=OLLAMA_BASE_URL, temperature=TEMPERATURE, reasoning=False, client_kwargs={'timeout': REQUEST_TIMEOUT_S})
qwen_planner = qwen_llm.with_structured_output(EntitySearchPlan, include_raw=True)
QWEN_PROMPT = DETAILED_PROMPT + '\nJSON Schema: ' + SCHEMA_JSON


def clean_terms(plan):
    if plan is None or plan.decision != 'candidate':
        return []
    terms, seen = [], set()
    for raw_term in plan.search_terms:
        term, key = raw_term.strip(), normalize_name(raw_term)
        if term and key and not key.isdigit() and key not in seen and len(term) <= 80:
            terms.append(term)
            seen.add(key)
    return terms[:2]


def expand_with_qwen(question, raw_name):
    started_at = time.perf_counter()
    try:
        result = qwen_planner.invoke([('system', QWEN_PROMPT), ('human', f'사용자 질문: {question}\n원문 기업 표현: {raw_name}')])
        plan = result.get('parsed')
        parsing_error = result.get('parsing_error')
        if plan is None:
            error = f'parsing_error: {parsing_error}' if parsing_error else 'parsed=None'
            return None, [], {}, error, (time.perf_counter() - started_at) * 1_000
        terms = clean_terms(plan)
        candidates = merge_candidates(*(merge_candidates(exact_candidates(term, 'llm_exact'), fuzzy_candidates(term, 'llm_fuzzy')) for term in terms))
        return plan, terms, candidates, None, (time.perf_counter() - started_at) * 1_000
    except Exception as exc:
        return None, [], {}, f'{type(exc).__name__}: {exc}', (time.perf_counter() - started_at) * 1_000

In [ ]:
def resolve_case(case, use_llm):
    started_at = time.perf_counter()
    raw_name = case['원문 기업 표현']
    raw_exact = exact_candidates(raw_name)
    raw_fuzzy = fuzzy_candidates(raw_name)
    base_candidates = merge_candidates(raw_exact, raw_fuzzy)
    best_fuzzy_score = max((candidate['score'] or 0 for candidate in raw_fuzzy.values()), default=0)
    key = normalize_name(raw_name)
    plan, terms, llm_candidates, error, llm_latency_ms = None, [], {}, None, None
    llm_called = False

    if key in GENERIC_GROUP_TERMS:
        state, selected_code = 'NEEDS_CONFIRMATION', None
    elif len(raw_exact) == 1:
        state, selected_code = 'RESOLVED', next(iter(raw_exact))
    else:
        if use_llm:
            llm_called = True
            plan, terms, llm_candidates, error, llm_latency_ms = expand_with_qwen(case['질문'], raw_name)
        candidates_after_llm = merge_candidates(base_candidates, llm_candidates)
        if llm_candidates:
            state, selected_code = 'NEEDS_CONFIRMATION', None
        elif best_fuzzy_score >= PROVISIONAL_CUTOFF:
            state, selected_code = 'NEEDS_CONFIRMATION', None
        else:
            state, selected_code = 'NOT_FOUND', None
        base_candidates = candidates_after_llm

    elapsed_ms = (time.perf_counter() - started_at) * 1_000
    expected_codes = {target_records[target]['corp_code'] for target in case['기대 대상']}
    candidate_codes = set(base_candidates)
    return {
        **case, '정책': 'B. local LLM expansion' if use_llm else 'A. deterministic',
        '상태': state, '선택 corp_code': selected_code, '후보': candidate_display(base_candidates),
        '후보 수': len(base_candidates), '최고 fuzzy 점수': best_fuzzy_score,
        'LLM 호출': llm_called, 'LLM 결정': plan.decision if plan else None,
        'LLM 검색어': terms, 'LLM 오류': error, 'LLM 시간(ms)': llm_latency_ms, '정책 시간(ms)': elapsed_ms,
        '정답 후보 포함': bool(expected_codes & candidate_codes) if expected_codes else None,
        '자동 확정 정답': state == 'RESOLVED' and selected_code in expected_codes,
        '위험 자동 확정': state == 'RESOLVED' and (case['기대 상태'] != 'RESOLVED' or selected_code not in expected_codes),
        '상태 정답': state == case['기대 상태'],
        'LLM 신규 기여': bool((expected_codes & set(llm_candidates)) - set(merge_candidates(raw_exact, raw_fuzzy))) if expected_codes else None,
        '미등록 LLM 후보 주입': case['기대 상태'] == 'NOT_FOUND' and bool(llm_candidates),
    }


rows = []
for case_number, case in enumerate(TEST_CASES, start=1):
    print(f'[{case_number}/{len(TEST_CASES)}] {case["원문 기업 표현"]!r}: A 실행', flush=True)
    rows.append(resolve_case(case, use_llm=False))
    print(f'[{case_number}/{len(TEST_CASES)}] {case["원문 기업 표현"]!r}: B 실행', flush=True)
    rows.append(resolve_case(case, use_llm=True))

results = pd.DataFrame(rows)
print(f'완료: 결과 {len(results)}행 / B의 LLM 호출 {int(results[results["정책"] == "B. local LLM expansion"]["LLM 호출"].sum())}회')
results[['정책', '원문 기업 표현', '유형', '기대 상태', '상태', '정답 후보 포함', '후보 수', 'LLM 검색어', 'LLM 오류']].sort_values(['원문 기업 표현', '정책'])

In [ ]:
def p95(values):
    return sorted(values)[ceil(len(values) * 0.95) - 1] if values else None


summary_rows = []
for policy, group in results.groupby('정책'):
    positive = group[group['기대 대상'].map(bool)]
    no_target = group[group['기대 상태'] == 'NOT_FOUND']
    auto = group[group['상태'] == 'RESOLVED']
    llm_rows = group[group['LLM 호출']]
    policy_times = group['정책 시간(ms)'].tolist()
    summary_rows.append({
        '정책': policy,
        'Candidate Recall': positive['정답 후보 포함'].mean(),
        'Auto-resolution Success Rate': positive['자동 확정 정답'].mean(),
        'Unsafe Selection Rate': auto['위험 자동 확정'].mean() if len(auto) else 0.0,
        'State Accuracy': group['상태 정답'].mean(),
        'Not-found Recall': (no_target['상태'] == 'NOT_FOUND').mean(),
        'False Not-found Rate': (positive['상태'] == 'NOT_FOUND').mean(),
        'Clarification Rate': (group['상태'] == 'NEEDS_CONFIRMATION').mean(),
        '후보 수 중앙값': group['후보 수'].median(),
        'LLM Expansion Contribution': positive['LLM 신규 기여'].mean(),
        'No-target LLM Candidate Injection Rate': no_target['미등록 LLM 후보 주입'].mean(),
        'LLM Call Rate': group['LLM 호출'].mean(),
        'LLM Structured-output Success Rate': (llm_rows['LLM 오류'].isna()).mean() if len(llm_rows) else None,
        'p50 ms': median(policy_times), 'p95 ms': p95(policy_times),
    })

summary = pd.DataFrame(summary_rows).set_index('정책')
display(summary)

changes = results.pivot(index='원문 기업 표현', columns='정책', values=['상태', '정답 후보 포함', 'LLM 검색어'])
display(changes)

## 판정 기준

- B는 A보다 Candidate Recall 또는 LLM Expansion Contribution을 높이고, Unsafe Selection Rate와 False Not-found Rate를 높이지 않을 때만 다음 API holdout 후보로 유지한다.
- 자동 오선택이 한 건이라도 있으면 해당 정책은 미채택한다. 단, 이 작은 validation에서 0이라고 실제 운영 오류율도 0이라고 말할 수는 없다.
- `NOT_FOUND`와 Candidate Recall이 충돌하거나, B의 개선이 LLM의 무관 후보 주입으로 만들어지면 결론을 보류한다.
- 이 결과는 로컬 Qwen의 정책 진단 결과다. 운영 API의 최종 근거는 정책을 고정한 뒤 별도 API holdout에서 확인한다.

실행 결과를 저장한 뒤 알려주면, 18번 HTML 보고서에 상단 요약(목적·핵심 결과·채택/미채택)과 하단 상세 결과를 기록한다.